# 🔴 미션 3 (선택) — 내가 고른 중의어의 어텐션을 본다

수업 `## 8` 에서 "배" 로 한 것을 **다른 중의어**로 한다.

후보: 눈(하늘/얼굴) · 밤(시간/먹는 것) · 말(동물/language) · 다리(신체/교량) · 차(마시는/타는)

## 낼 것

1. **bertviz 캡처 1장** — 두 문맥에서 어텐션이 달라 보이는 장면
2. 한 줄 — **몇 층의 어느 헤드**에서 차이가 잘 보였나

In [1]:
import sys
from pathlib import Path

import torch
from bertviz import head_view
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
viz_model = AutoModel.from_pretrained(
    "klue/bert-base", output_attentions=True, attn_implementation="eager"
)
viz_model.eval()

/home/student/llm-practice/c2-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2766.01it/s]
[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be igno

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

## 1. 중의어와 문장 2개를 고른다

✍️ **중의어 하나를 골라, 뜻이 다른 문장 2개를 만든다.**

In [2]:
내문장들 = [
    "점심을 너무 많이 먹어서 배가 빵빵하다.",       # ✍️ 바꾼다 — 뜻 1
    "섬에 들어가기 위해 선착장에서 배를 탔다.",     # ✍️ 바꾼다 — 뜻 2
    "제사상에 올릴 크고 달콤한 배를 샀다."
]

## 2. 먼저 확인 — 내 중의어가 한 조각으로 남아 있는가

단어가 조각나면(`##` 로 쪼개지면) 비교가 어렵다. 쪼개진다면 문장을 조금 바꿔 보라.

In [3]:
for sent in 내문장들:
    print(tokenizer.tokenize(sent))

['점심', '##을', '너무', '많이', '먹', '##어', '##서', '배', '##가', '빵빵', '##하', '##다', '.']
['섬', '##에', '들어가', '##기', '위해', '선착장', '##에서', '배', '##를', '탔', '##다', '.']
['제사', '##상', '##에', '올릴', '크', '##고', '달콤', '##한', '배', '##를', '샀', '##다', '.']


## 3. 어텐션을 그린다 — 수업 `## 8` 그대로

In [4]:
for i, sent in enumerate(내문장들, start=1):
    inputs = tokenizer(sent, return_tensors="pt")
    with torch.no_grad():
        outputs = viz_model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    # attentions: 층마다 (1, 헤드, 토큰, 토큰) · html_action="return": HTML로 돌려받아 저장
    viz_html = head_view(outputs.attentions, tokens, html_action="return")

    out_file = Path(f"mission3_{i}.html")
    out_file.write_text(viz_html.data, encoding="utf-8")
    print(f"[{i}] {sent}  →  {out_file} 저장 (브라우저로 연다)")

    if "ipykernel" in sys.modules:
        from IPython.display import display

        display(viz_html)

[1] 점심을 너무 많이 먹어서 배가 빵빵하다.  →  mission3_1.html 저장 (브라우저로 연다)


[2] 섬에 들어가기 위해 선착장에서 배를 탔다.  →  mission3_2.html 저장 (브라우저로 연다)


[3] 제사상에 올릴 크고 달콤한 배를 샀다.  →  mission3_3.html 저장 (브라우저로 연다)


## 4. 찾는다

층(0~11)을 바꿔 가며 **내 중의어의 줄**을 따라가라.
두 문장에서 그 단어가 **다른 곳을 강하게 보는** 층·헤드를 찾으면 캡처.

## ✍️ 제출 — 한 줄

> ____ 층의 ____ 번째 헤드에서, "____"가 문장 1에서는 ____ 를, 문장 2에서는 ____ 를 강하게 봤다.

**6**층의 **9**번째(인덱스) 헤드에서, "**배**"가 문장 1에서는 **빵빵**을, 문장 2에서는 **탔**을 강하게 봤다.

- 문장1 (점심을 너무 많이 먹어서 배가 빵빵하다): "배" → "빵빵" 가중치 0.72
- 문장2 (섬에 들어가기 위해 선착장에서 배를 탔다): "배" → "탔" 가중치 0.90
- 문장3 (제사상에 올릴 크고 달콤한 배를 샀다): "배" → "샀" 가중치 0.63

> ⚠️ 모든 층·헤드에서 차이가 보이는 것이 아니다. 안 보이는 층이 더 많다 — 그게 정상이다.
> "이 헤드는 중의어 담당"이라고 단정할 근거도 없다. **본 것까지만 적는다.**